# Hands-on PySpark Dasar
**Tujuan:** berlatih membuat DataFrame, melihat skema, memilih dan memfilter kolom/baris, menambah kolom, melakukan agregasi, mengurutkan data, join, dan Spark SQL.

**Kasus:** analisis transaksi penjualan sederhana. Notebook ini bisa dijalankan di Google Colab atau Jupyter. Jalankan sel secara berurutan.

## 1. Instalasi
Jalankan sel berikut jika PySpark belum tersedia di lingkungan notebook. Di Google Colab biasanya perlu dijalankan sekali per sesi.

In [1]:
%pip -q install pyspark


[notice] A new release of pip is available: 24.0 -> 26.2.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 2. Membuat SparkSession
`SparkSession` adalah titik masuk untuk bekerja dengan DataFrame dan Spark SQL.

In [2]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, sum as spark_sum, avg, count

# Only for spark master
# spark = (
#     SparkSession.builder
#     .appName("HandsOnPySparkDasar")
#     .master("local[*]")
#     .getOrCreate()
# )

# Only for spark connect
spark = (
    SparkSession.builder
    .appName("HandsOnPySparkDasar")
    .master("local[*]")
    .getOrCreate()
)

print("Spark version:", spark.version)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/10/13 20:29:39 WARN Utils: Your hostname, Rezas-MacBook-Pro-M4-Pro.local, resolves to a loopback address: 127.0.2.2; using 192.168.18.137 instead (on interface en0)
26/10/13 20:29:39 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/Users/fahmialfareza/Code/learning/data-engineering/fullstack-bangalore-advanced-data-engineering/17-adv-data-engineering/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
26/10/13 20:29:40 WARN NativeCodeLoader: Unable to load native-hadoop librar

Spark version: 4.2.0


## 3. Membuat DataFrame contoh
Data dibuat langsung di notebook agar latihan tidak membutuhkan file eksternal.

In [3]:
data = [
    (1, "Andi", "Jakarta", "Laptop", 1, 7500000, "Selesai"),
    (2, "Budi", "Bandung", "Mouse", 2, 150000, "Selesai"),
    (3, "Citra", "Jakarta", "Keyboard", 1, 350000, "Diproses"),
    (4, "Dina", "Surabaya", "Monitor", 1, 2200000, "Selesai"),
    (5, "Eko", "Bandung", "Laptop", 1, 7500000, "Dibatalkan"),
    (6, "Fitri", "Jakarta", "Mouse", 3, 150000, "Selesai"),
    (7, "Gilang", "Surabaya", "Keyboard", 2, 350000, "Diproses"),
    (8, "Hana", "Bandung", "Monitor", 1, 2200000, "Selesai"),
]

kolom = ["id_transaksi", "pelanggan", "kota", "produk", "jumlah", "harga", "status"]
df = spark.createDataFrame(data, kolom)

df.show()

+------------+---------+--------+--------+------+-------+----------+
|id_transaksi|pelanggan|    kota|  produk|jumlah|  harga|    status|
+------------+---------+--------+--------+------+-------+----------+
|           1|     Andi| Jakarta|  Laptop|     1|7500000|   Selesai|
|           2|     Budi| Bandung|   Mouse|     2| 150000|   Selesai|
|           3|    Citra| Jakarta|Keyboard|     1| 350000|  Diproses|
|           4|     Dina|Surabaya| Monitor|     1|2200000|   Selesai|
|           5|      Eko| Bandung|  Laptop|     1|7500000|Dibatalkan|
|           6|    Fitri| Jakarta|   Mouse|     3| 150000|   Selesai|
|           7|   Gilang|Surabaya|Keyboard|     2| 350000|  Diproses|
|           8|     Hana| Bandung| Monitor|     1|2200000|   Selesai|
+------------+---------+--------+--------+------+-------+----------+



## 4. Memahami struktur data
- `show()` menampilkan beberapa baris.
- `printSchema()` menampilkan nama kolom dan tipe datanya.
- `count()` menghitung jumlah baris.

In [4]:
df.printSchema()
print("Jumlah baris:", df.count())

root
 |-- id_transaksi: long (nullable = true)
 |-- pelanggan: string (nullable = true)
 |-- kota: string (nullable = true)
 |-- produk: string (nullable = true)
 |-- jumlah: long (nullable = true)
 |-- harga: long (nullable = true)
 |-- status: string (nullable = true)

Jumlah baris: 8


## 5. Memilih kolom (`select`)
Tampilkan hanya nama pelanggan, kota, dan produk.

In [5]:
df.select("pelanggan", "kota", "produk").show()

+---------+--------+--------+
|pelanggan|    kota|  produk|
+---------+--------+--------+
|     Andi| Jakarta|  Laptop|
|     Budi| Bandung|   Mouse|
|    Citra| Jakarta|Keyboard|
|     Dina|Surabaya| Monitor|
|      Eko| Bandung|  Laptop|
|    Fitri| Jakarta|   Mouse|
|   Gilang|Surabaya|Keyboard|
|     Hana| Bandung| Monitor|
+---------+--------+--------+



## 6. Memfilter baris (`filter`)
Tampilkan transaksi yang berstatus `Selesai`.

In [6]:
df.filter(col("status") == "Selesai").show()

+------------+---------+--------+-------+------+-------+-------+
|id_transaksi|pelanggan|    kota| produk|jumlah|  harga| status|
+------------+---------+--------+-------+------+-------+-------+
|           1|     Andi| Jakarta| Laptop|     1|7500000|Selesai|
|           2|     Budi| Bandung|  Mouse|     2| 150000|Selesai|
|           4|     Dina|Surabaya|Monitor|     1|2200000|Selesai|
|           6|    Fitri| Jakarta|  Mouse|     3| 150000|Selesai|
|           8|     Hana| Bandung|Monitor|     1|2200000|Selesai|
+------------+---------+--------+-------+------+-------+-------+



## 7. Menambahkan kolom baru (`withColumn`)
Hitung nilai transaksi dengan rumus `jumlah × harga`. Kolom lama tetap dipertahankan.

In [7]:
df_nilai = df.withColumn("nilai_transaksi", col("jumlah") * col("harga"))
df_nilai.show()

+------------+---------+--------+--------+------+-------+----------+---------------+
|id_transaksi|pelanggan|    kota|  produk|jumlah|  harga|    status|nilai_transaksi|
+------------+---------+--------+--------+------+-------+----------+---------------+
|           1|     Andi| Jakarta|  Laptop|     1|7500000|   Selesai|        7500000|
|           2|     Budi| Bandung|   Mouse|     2| 150000|   Selesai|         300000|
|           3|    Citra| Jakarta|Keyboard|     1| 350000|  Diproses|         350000|
|           4|     Dina|Surabaya| Monitor|     1|2200000|   Selesai|        2200000|
|           5|      Eko| Bandung|  Laptop|     1|7500000|Dibatalkan|        7500000|
|           6|    Fitri| Jakarta|   Mouse|     3| 150000|   Selesai|         450000|
|           7|   Gilang|Surabaya|Keyboard|     2| 350000|  Diproses|         700000|
|           8|     Hana| Bandung| Monitor|     1|2200000|   Selesai|        2200000|
+------------+---------+--------+--------+------+-------+--------

## 8. Agregasi (`groupBy` dan `agg`)
Hitung total nilai transaksi dan jumlah transaksi untuk setiap kota. Latihan ini memakai seluruh status transaksi.

In [8]:
ringkasan_kota = (
    df_nilai.groupBy("kota")
    .agg(
        spark_sum("nilai_transaksi").alias("total_nilai"),
        count("*").alias("jumlah_transaksi")
    )
    .orderBy(col("total_nilai").desc())
)

ringkasan_kota.show()

+--------+-----------+----------------+
|    kota|total_nilai|jumlah_transaksi|
+--------+-----------+----------------+
| Bandung|   10000000|               3|
| Jakarta|    8300000|               3|
|Surabaya|    2900000|               2|
+--------+-----------+----------------+



## 9. Mengurutkan data (`orderBy`)
Tampilkan transaksi dari nilai terbesar ke terkecil.

In [9]:
df_nilai.orderBy(col("nilai_transaksi").desc()).show()

+------------+---------+--------+--------+------+-------+----------+---------------+
|id_transaksi|pelanggan|    kota|  produk|jumlah|  harga|    status|nilai_transaksi|
+------------+---------+--------+--------+------+-------+----------+---------------+
|           5|      Eko| Bandung|  Laptop|     1|7500000|Dibatalkan|        7500000|
|           1|     Andi| Jakarta|  Laptop|     1|7500000|   Selesai|        7500000|
|           4|     Dina|Surabaya| Monitor|     1|2200000|   Selesai|        2200000|
|           8|     Hana| Bandung| Monitor|     1|2200000|   Selesai|        2200000|
|           7|   Gilang|Surabaya|Keyboard|     2| 350000|  Diproses|         700000|
|           6|    Fitri| Jakarta|   Mouse|     3| 150000|   Selesai|         450000|
|           3|    Citra| Jakarta|Keyboard|     1| 350000|  Diproses|         350000|
|           2|     Budi| Bandung|   Mouse|     2| 150000|   Selesai|         300000|
+------------+---------+--------+--------+------+-------+--------

## 10. Join dua DataFrame
Gabungkan transaksi dengan informasi kategori produk.

In [10]:
data_produk = [
    ("Laptop", "Elektronik"),
    ("Mouse", "Aksesori"),
    ("Keyboard", "Aksesori"),
    ("Monitor", "Elektronik"),
]
df_produk = spark.createDataFrame(data_produk, ["produk", "kategori"])

df_join = df_nilai.join(df_produk, on="produk", how="left")
df_join.show()

+--------+------------+---------+--------+------+-------+----------+---------------+----------+
|  produk|id_transaksi|pelanggan|    kota|jumlah|  harga|    status|nilai_transaksi|  kategori|
+--------+------------+---------+--------+------+-------+----------+---------------+----------+
|  Laptop|           1|     Andi| Jakarta|     1|7500000|   Selesai|        7500000|Elektronik|
|   Mouse|           2|     Budi| Bandung|     2| 150000|   Selesai|         300000|  Aksesori|
|Keyboard|           3|    Citra| Jakarta|     1| 350000|  Diproses|         350000|  Aksesori|
| Monitor|           4|     Dina|Surabaya|     1|2200000|   Selesai|        2200000|Elektronik|
|  Laptop|           5|      Eko| Bandung|     1|7500000|Dibatalkan|        7500000|Elektronik|
|   Mouse|           6|    Fitri| Jakarta|     3| 150000|   Selesai|         450000|  Aksesori|
|Keyboard|           7|   Gilang|Surabaya|     2| 350000|  Diproses|         700000|  Aksesori|
| Monitor|           8|     Hana| Bandun

## 11. Spark SQL
DataFrame dapat didaftarkan sebagai temporary view, lalu dianalisis menggunakan SQL.

In [11]:
df_nilai.createOrReplaceTempView("transaksi")

spark.sql("""
    SELECT kota,
           COUNT(*) AS jumlah_transaksi,
           SUM(jumlah * harga) AS total_nilai
    FROM transaksi
    WHERE status = 'Selesai'
    GROUP BY kota
    ORDER BY total_nilai DESC
""").show()

+--------+----------------+-----------+
|    kota|jumlah_transaksi|total_nilai|
+--------+----------------+-----------+
| Jakarta|               2|    7950000|
| Bandung|               2|    2500000|
|Surabaya|               1|    2200000|
+--------+----------------+-----------+



## 12. Menyimpan hasil ke Parquet (opsional)
Parquet adalah format kolumnar yang umum dipakai untuk data analitik. Sel ini membuat folder `output_transaksi_parquet` di direktori kerja.

In [12]:
# Jalankan jika ingin mencoba menyimpan hasil
df_nilai.write.mode("overwrite").parquet("output_transaksi_parquet")

# Membaca kembali hasilnya
df_baca = spark.read.parquet("output_transaksi_parquet")
df_baca.show(5)

26/10/13 20:30:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/10/13 20:30:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 84.44% for 9 writers
26/10/13 20:30:28 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers


+------------+---------+--------+--------+------+-------+----------+---------------+
|id_transaksi|pelanggan|    kota|  produk|jumlah|  harga|    status|nilai_transaksi|
+------------+---------+--------+--------+------+-------+----------+---------------+
|           7|   Gilang|Surabaya|Keyboard|     2| 350000|  Diproses|         700000|
|           3|    Citra| Jakarta|Keyboard|     1| 350000|  Diproses|         350000|
|           4|     Dina|Surabaya| Monitor|     1|2200000|   Selesai|        2200000|
|           5|      Eko| Bandung|  Laptop|     1|7500000|Dibatalkan|        7500000|
|           8|     Hana| Bandung| Monitor|     1|2200000|   Selesai|        2200000|
+------------+---------+--------+--------+------+-------+----------+---------------+
only showing top 5 rows


## 13. Latihan mandiri
Coba kerjakan sendiri sebelum melihat contoh solusi.

1. Tampilkan transaksi dengan `harga` lebih dari 1.000.000.
2. Tampilkan transaksi yang statusnya `Diproses`.
3. Tambahkan kolom `diskon` sebesar 10% dari `nilai_transaksi`.
4. Hitung total nilai transaksi untuk setiap `produk`.
5. Cari kota dengan jumlah transaksi terbanyak.
6. Gunakan Spark SQL untuk menampilkan transaksi berstatus `Selesai` dengan nilai transaksi di atas 500.000.

**Petunjuk:** gunakan `filter`, `withColumn`, `groupBy`, `agg`, `orderBy`, dan `spark.sql`.

## 14. Contoh solusi latihan
Buka bagian ini setelah mencoba mengerjakan latihan secara mandiri.

In [13]:
# 1. Harga > 1.000.000
df.filter(col("harga") > 1_000_000).show()

# 2. Status Diproses
df.filter(col("status") == "Diproses").show()

# 3. Diskon 10% dari nilai transaksi
df_diskon = (
    df_nilai
    .withColumn("diskon", col("nilai_transaksi") * 0.10)
    .withColumn("nilai_setelah_diskon", col("nilai_transaksi") * 0.90)
)
df_diskon.show()

# 4. Total nilai transaksi per produk
df_nilai.groupBy("produk").agg(
    spark_sum("nilai_transaksi").alias("total_nilai")
).orderBy(col("total_nilai").desc()).show()

# 5. Kota dengan transaksi terbanyak
df.groupBy("kota").agg(
    count("*").alias("jumlah_transaksi")
).orderBy(col("jumlah_transaksi").desc()).show()

# 6. Spark SQL: transaksi selesai di atas 500.000
spark.sql("""
    SELECT *
    FROM transaksi
    WHERE status = 'Selesai' AND (jumlah * harga) > 500000
""").show()

+------------+---------+--------+-------+------+-------+----------+
|id_transaksi|pelanggan|    kota| produk|jumlah|  harga|    status|
+------------+---------+--------+-------+------+-------+----------+
|           1|     Andi| Jakarta| Laptop|     1|7500000|   Selesai|
|           4|     Dina|Surabaya|Monitor|     1|2200000|   Selesai|
|           5|      Eko| Bandung| Laptop|     1|7500000|Dibatalkan|
|           8|     Hana| Bandung|Monitor|     1|2200000|   Selesai|
+------------+---------+--------+-------+------+-------+----------+

+------------+---------+--------+--------+------+------+--------+
|id_transaksi|pelanggan|    kota|  produk|jumlah| harga|  status|
+------------+---------+--------+--------+------+------+--------+
|           3|    Citra| Jakarta|Keyboard|     1|350000|Diproses|
|           7|   Gilang|Surabaya|Keyboard|     2|350000|Diproses|
+------------+---------+--------+--------+------+------+--------+

+------------+---------+--------+--------+------+-------+-

## 15. Penutup
Yang dipelajari:
- Membuat `SparkSession` dan DataFrame.
- Melihat skema, memilih kolom, dan memfilter baris.
- Membuat kolom perhitungan.
- Agregasi, pengurutan, dan join.
- Menjalankan query dengan Spark SQL.
- Menyimpan dan membaca data Parquet.

Untuk menutup sesi Spark, jalankan sel berikut.

In [14]:
spark.stop()